## Evaluation
**Package:** `conditional_embedding_model`
**Source notebook:** `train/TrainNS_Eval.ipynb`
**Purpose:** Load a trained checkpoint and run quantitative evaluation (Hit@k, precision-recall,
ROC), a recommendation demo, and a failure case analysis.

## 1. Setup & Imports
Training-time evaluation utilities (temperature-scaled metrics, evaluate_model) come from
`conditional_embedding_model.training`. Post-training evaluation routines (optimised Hit@k,
ROC/PR plotting, t-SNE) come from `evaluation/evaluate.py` at the project root.

> **Note:** `evaluation/evaluate.py` is not part of the inner Python package
> `conditional_embedding_model/`. Import it as `from evaluation.evaluate import ...`
> after adding the project root to `sys.path`.

In [ ]:
import os
import sys
import torch
from torch import nn
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, precision_recall_curve

sys.path.insert(0, os.path.abspath(".."))

from conditional_embedding_model.training import (
    define_model,
    prediction_temperature,
    evaluate_model,
    evaluate_model_topk,
    evaluate_metrics,
)
from conditional_embedding_model.data import load_data_grasp, batchify_wrapper

# evaluation module lives at <project_root>/evaluation/evaluate.py
from evaluation.evaluate import (
    evaluate_model_topk_optimized,
    evaluate_metrics_optimized,
    plot_roc_curve,
    plot_precision_recall_curve,
    graph_2dspace,
    graph_3dspace,
    optimize_threshold,
)

print("Imports OK")

## 2. Configuration
Change only the values in `CONFIG` — all downstream cells read from it.
Set `checkpoint_path` to the `.pth` file produced by `02_training.ipynb`.

In [ ]:
CONFIG = {
    "data_root":      "../config/data",
    "dataset_file":   "CoopGrasping-v6_dataset.pkl",
    "checkpoint_path": "../config/weights/model.pth",  # <-- trained checkpoint
    # model architecture — must match the checkpoint
    "input_size":     4,
    "embedding_size": 44,
    "dropout":        0.02,
    "model_version":  "v4",
    # dataloader
    "batch_size":     32,
    "max_length":     100,
    "multiplier_factor": 1.0,
    # evaluation
    "top_k_values":   [1, 3, 5, 7],
    "device": "cuda:0" if torch.cuda.is_available() else "cpu",
}

print("Device     :", CONFIG["device"])
print("Checkpoint :", CONFIG["checkpoint_path"])

## 3. Load Model
Build the model architecture, load checkpoint weights, and switch to eval mode.
This cell also builds the test DataLoader used throughout the remaining sections.

In [ ]:
dataset_path = os.path.join(CONFIG["data_root"], CONFIG["dataset_file"])
_, _, test_ds, feature_struct = load_data_grasp(
    CONFIG["batch_size"],
    CONFIG["multiplier_factor"],
    dataset_path,
    CONFIG["max_length"],
)
test_dl = DataLoader(
    test_ds,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    collate_fn=batchify_wrapper(CONFIG["max_length"]),
)

net = define_model(
    input_size=CONFIG["input_size"],
    embedding_size=CONFIG["embedding_size"],
    feature_structure=feature_struct,
    dropout=CONFIG["dropout"],
    map_flag=True,
    device=CONFIG["device"],
    model_version=CONFIG["model_version"],
)
net.load_state_dict(
    torch.load(CONFIG["checkpoint_path"], map_location=CONFIG["device"])
)
net = net.to(CONFIG["device"]).eval()
print("Checkpoint loaded and model set to eval mode.")

## 4. Quantitative Evaluation
Compute binary classification metrics (Accuracy, Precision, Recall, F1, AUC-ROC, PR-AUC)
and Hit@k retrieval accuracy. Results are shown as a pandas DataFrame.

`evaluate_model` from `training` uses temperature-scaled logits (consistent with training).
`evaluate_model_topk_optimized` from `evaluation` uses cosine similarity and correctly
excludes negative-only rows from the denominator.

> **TODO:** NDCG@k is not currently implemented in the package. Add it to
> `evaluation/evaluate.py` and import it here once available.

In [ ]:
# -- Binary classification metrics (temperature-scaled, consistent with training) --
print("=== Binary Classification Metrics ===")
metrics, pr_auc, best_thr = evaluate_model(
    net,
    test_dl,
    device=CONFIG["device"],
    threshold=None,   # searches for optimal F1 threshold
    verbose=True,
    method="weighted",
)
accuracy, precision, recall, f1, auc_roc = metrics
print(f"PR-AUC: {pr_auc:.4f}   Best threshold: {best_thr:.2f}")

print()
# -- Hit@k (temperature-scaled, from training module) --
print("=== Hit@k (training.evaluate_model_topk) ===")
topk_acc = evaluate_model_topk(
    net,
    test_dl,
    device=CONFIG["device"],
    verbose=True,
    k_list=CONFIG["top_k_values"],
)

print()
# -- Hit@k optimised (cosine similarity, from evaluation module) --
print("=== Hit@k optimised (evaluation.evaluate_model_topk_optimized) ===")
topk_counts = evaluate_model_topk_optimized(
    net,
    test_dl,
    device=CONFIG["device"],
    verbose=True,
    cosine=True,
)

In [ ]:
# Build a tidy metrics DataFrame for display
rows = [
    {"Metric": "Accuracy",      "Value": accuracy},
    {"Metric": "Precision",     "Value": precision},
    {"Metric": "Recall",        "Value": recall},
    {"Metric": "F1 (weighted)", "Value": f1},
    {"Metric": "AUC-ROC",       "Value": auc_roc},
    {"Metric": "PR-AUC",        "Value": pr_auc},
    {"Metric": "Threshold",     "Value": best_thr},
]
for k in CONFIG["top_k_values"]:
    rows.append({"Metric": f"Hit@{k}", "Value": topk_acc[k]})

df_metrics = pd.DataFrame(rows).set_index("Metric").round(4)
print(df_metrics.to_string())

### ROC and Precision-Recall curves
Collect raw sigmoid probabilities from a forward pass over the test set, then plot
ROC and PR curves with matplotlib and sklearn.

In [ ]:
# Collect all predictions and labels over the test set
all_probs, all_labels = [], []
with torch.no_grad():
    for center, ctx_neg, mask, label, feature in test_dl:
        center, ctx_neg, mask, label, feature = [
            x.to(CONFIG["device"]) for x in [center, ctx_neg, mask, label, feature]
        ]
        logits = prediction_temperature(
            center, ctx_neg, net[0], net[1], feature, temperature=0.07
        )
        logits = logits.view_as(label)
        # Keep only valid (non-padded) positions
        valid = mask.bool()
        all_probs.append(logits[valid].sigmoid().cpu())
        all_labels.append(label[valid].cpu())

probs_np  = torch.cat(all_probs).numpy()
labels_np = torch.cat(all_labels).numpy()

# %%
# ROC curve
fpr, tpr, _ = roc_curve(labels_np, probs_np)
roc_auc_val = auc(fpr, tpr)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC (AUC={roc_auc_val:.3f})")
axes[0].plot([0, 1], [0, 1], color="navy", lw=1.5, linestyle="--")
axes[0].set_xlabel("False Positive Rate", fontsize=12)
axes[0].set_ylabel("True Positive Rate", fontsize=12)
axes[0].set_title("ROC Curve", fontsize=14)
axes[0].legend(fontsize=11)

# PR curve
prec, rec, _ = precision_recall_curve(labels_np, probs_np)
pr_auc_val = auc(rec, prec)

axes[1].plot(rec, prec, color="blue", lw=2, label=f"PR (AUC={pr_auc_val:.3f})")
axes[1].set_xlabel("Recall", fontsize=12)
axes[1].set_ylabel("Precision", fontsize=12)
axes[1].set_title("Precision-Recall Curve", fontsize=14)
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.show()

## 5. Recommendation Demo
Given a query center from the test set, retrieve the top-k most compatible grasping
configurations ranked by temperature-scaled similarity. The table shows the predicted
robot indices and their raw logit scores alongside the ground-truth positive indices.

In [ ]:
k_demo = CONFIG["top_k_values"][2]  # k=5

for batch in test_dl:
    center, ctx_neg, mask, label, feature = [x.to(CONFIG["device"]) for x in batch]
    break

with torch.no_grad():
    logits = prediction_temperature(
        center, ctx_neg, net[0], net[1], feature, temperature=0.07
    )
    logits[mask == 0] = -float("inf")
    scores, top_k_idx = logits.topk(k_demo, dim=1)

print(f"Top-{k_demo} recommendations for the first 5 query samples")
print(f"{'Query':>6}  {'Top-k indices':>25}  {'Scores':>30}  {'Ground-truth positives':>25}")
print("-" * 92)
for i in range(min(5, center.size(0))):
    idx_list   = top_k_idx[i].cpu().tolist()
    score_list = [f"{s:.3f}" for s in scores[i].cpu().tolist()]
    true_idx   = label[i].nonzero(as_tuple=True)[0].tolist()
    hit        = any(ix in true_idx for ix in idx_list)
    marker     = "HIT" if hit else "MISS"
    print(
        f"{i:>6}  {str(idx_list):>25}  {str(score_list):>30}"
        f"  {str(true_idx):>25}  [{marker}]"
    )

## 6. Failure Case Analysis
Scan the test set for samples where the model's top-1 prediction is wrong (the highest-
scoring index is not in the ground-truth positive set). Display query center, the top-1
and top-5 predictions, and the actual positive indices for 3 such failure cases.

In [ ]:
failures = []

with torch.no_grad():
    for batch in test_dl:
        center, ctx_neg, mask, label, feature = [
            x.to(CONFIG["device"]) for x in batch
        ]
        logits = prediction_temperature(
            center, ctx_neg, net[0], net[1], feature, temperature=0.07
        )
        logits[mask == 0] = -float("inf")
        top1_idx = logits.argmax(dim=1)  # (B,)
        top5_idx = logits.topk(5, dim=1).indices

        for i in range(center.size(0)):
            true_set = set(label[i].nonzero(as_tuple=True)[0].tolist())
            pred_1   = top1_idx[i].item()
            if true_set and pred_1 not in true_set:
                failures.append({
                    "center":    center[i].squeeze().cpu().item(),
                    "top1_pred": pred_1,
                    "top5_pred": top5_idx[i].cpu().tolist(),
                    "true_pos":  sorted(true_set),
                    "logit_top1": logits[i, pred_1].cpu().item(),
                })
            if len(failures) >= 3:
                break
        if len(failures) >= 3:
            break

print(f"Found {len(failures)} failure case(s) to display:")
for n, case in enumerate(failures[:3]):
    print(f"\n--- Failure case {n+1} ---")
    print(f"  Query center index   : {case['center']}")
    print(f"  Top-1 prediction     : {case['top1_pred']}  (logit={case['logit_top1']:.3f})")
    print(f"  Top-5 predictions    : {case['top5_pred']}")
    print(f"  Ground-truth positives: {case['true_pos']}")